# Task 2, Part A: Dense, Tiled Online-Softmax and SDPA Causal Attention

Synthetic Q, K, V with B=2, H=4, d_h=64, L in {256, ..., 8192}, FP16 and FP32.
* **A1 dense:** explicit QK^T, causal mask, stable FP32 softmax, x V (materialises L x L scores).
* **A2 tiled:** exact blockwise forward with the online-softmax update; never materialises L x L. The backward
  is ordinary autograd through the Python loop (stores per-tile intermediates) -- not a fused FlashAttention kernel.
* **A3 SDPA:** `torch.nn.functional.scaled_dot_product_attention`, default dispatch and each forced backend; the
  dispatched kernel is read from the profiler (not assumed).

Reference: dense attention in FP32 (FP64 for L <= 1024) on identical tensors. Ten warmed-up, CUDA-synchronised
repetitions per condition; OOM / unsupported conditions are recorded, never forced.

In [ ]:
# --- Setup: locate (or clone) the PA4 repo, pin library versions, prepare data -------------
import os, sys, subprocess, importlib.metadata as md
REPO_URL = "https://github.com/moizejaz-31/ATML-PA4.git"   # change if your repo lives elsewhere

def _find_root(d):
    d = os.path.abspath(d)
    while True:
        if os.path.isdir(os.path.join(d, "shared")) and os.path.isdir(os.path.join(d, "task2_attention_kv")):
            return d
        if os.path.dirname(d) == d:
            return None
        d = os.path.dirname(d)

ROOT = _find_root(os.getcwd())
if ROOT is None:                                   # fresh Kaggle / Colab runtime
    subprocess.run(["git", "clone", "-q", REPO_URL, "ATML-PA4"], check=True)
    ROOT = os.path.abspath("ATML-PA4")
os.chdir(ROOT)
sys.path.insert(0, ROOT)

def _ver(p):
    try:
        return md.version(p)
    except md.PackageNotFoundError:
        return None

if _ver("transformers") != "4.57.1" or _ver("huggingface_hub") != "0.36.2":
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers==4.57.1", "huggingface_hub==0.36.2",
                    "datasets>=3,<5"], check=True)
    print(">>> Installed pinned versions. Restart the kernel, then run this cell again.")

from shared.src.nb import init_notebook
ctx = init_notebook("task2", need_public=False, need_prefix_banks=False)

In [ ]:
import gc, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from shared.src.io import save_json, write_jsonl, read_jsonl
from shared.src.plotting import save_figure
from shared.src.timing import benchmark, peak_memory, reset_peak
from task2_attention_kv.src import attention_ops as A

assert torch.cuda.is_available()
DEV = "cuda"
B, H, D = 2, 4, 64
LENGTHS = [256, 512, 1024, 2048, 4096, 8192]
DTYPES = {"fp16": torch.float16, "fp32": torch.float32}
IMPLS = ["dense", "tiled", "sdpa_default", "sdpa_flash", "sdpa_efficient", "sdpa_math", "sdpa_cudnn"]
OUT = ctx.results / "attention"
OUT.mkdir(exist_ok=True)

def make_qkv(L, dtype, seed=6304):
    g = torch.Generator(device=DEV).manual_seed(seed + L)
    return [torch.randn(B, H, L, D, device=DEV, dtype=dtype, generator=g) for _ in range(3)]

def free():
    gc.collect(); torch.cuda.empty_cache()

## Which SDPA backend is dispatched? (per dtype / length, forward and backward)

In [ ]:
rows = []
for dn, dt in DTYPES.items():
    for L in (256, 2048, 8192):
        q, k, v = [t.requires_grad_() for t in make_qkv(L, dt)]
        for be in A.BACKENDS:
            try:
                fwd = A.detect_sdpa_backend(q, k, v, be)
                bwd = A.detect_sdpa_backend(q, k, v, be, grad=True)
                rows.append(dict(dtype=dn, L=L, requested=be, forward=fwd, backward=bwd, status="ok"))
            except Exception as e:
                rows.append(dict(dtype=dn, L=L, requested=be, forward=None, backward=None, status=f"unsupported: {str(e)[:90]}"))
        del q, k, v; free()
backend_df = pd.DataFrame(rows)
backend_df.to_csv(OUT / "sdpa_backends.csv", index=False)
backend_df

## A1-A3 correctness: output and gradient error vs the dense FP32 reference

In [ ]:
def ref_out_grads(q, k, v, go, L):
    rd = torch.float64 if L <= 1024 else torch.float32
    qr, kr, vr = [t.detach().to(rd).requires_grad_() for t in (q, k, v)]
    o = A.dense_attention(qr, kr, vr)
    o.backward(go.to(rd))
    return o.detach(), [t.grad.detach() for t in (qr, kr, vr)]

err_rows = []
for L in LENGTHS:
    try:
        q32, k32, v32 = make_qkv(L, torch.float32)
        go = torch.randn_like(q32)
        ref, ref_g = ref_out_grads(q32, k32, v32, go, L)
    except torch.cuda.OutOfMemoryError:
        err_rows.append(dict(L=L, impl="reference", status="OOM")); free(); continue
    for dn, dt in DTYPES.items():
        for impl in IMPLS:
            row = dict(L=L, dtype=dn, impl=impl)
            try:
                q, k, v = [t.detach().to(dt).requires_grad_() for t in (q32, k32, v32)]
                o = A.get_impl(impl)(q, k, v)
                o.backward(go.to(dt))
                row.update(out_max_abs=float((o.double() - ref.double()).abs().max()),
                           out_mean_abs=float((o.double() - ref.double()).abs().mean()),
                           finite=bool(torch.isfinite(o).all()), status="ok")
                for nm, gg, rg in zip(("dq", "dk", "dv"), (q.grad, k.grad, v.grad), ref_g):
                    row[f"{nm}_max_abs"] = float((gg.double() - rg.double()).abs().max())
                    row[f"{nm}_rel"] = float((gg.double() - rg.double()).norm() / rg.double().norm())
            except torch.cuda.OutOfMemoryError:
                row["status"] = "OOM"
            except RuntimeError as e:
                row["status"] = f"unsupported: {str(e)[:80]}"
            err_rows.append(row)
            q = k = v = o = None; free()
    del q32, k32, v32, ref, ref_g; free()
err_df = pd.DataFrame(err_rows)
err_df.to_csv(OUT / "accuracy_vs_reference.csv", index=False)
err_df.pivot_table(index=["impl", "dtype"], columns="L", values="out_max_abs")

In [ ]:
err_df.pivot_table(index=["impl", "dtype"], columns="L", values="dq_rel")

## Causality and edge cases

In [ ]:
edge = []
L = 512
for dn, dt in DTYPES.items():
    q, k, v = make_qkv(L, dt)
    k2, v2 = k.clone(), v.clone(); k2[:, :, 300:] += 3; v2[:, :, 300:] -= 3
    for impl in IMPLS:
        try:
            f = A.get_impl(impl)
            same = torch.equal(f(q, k, v)[:, :, :300], f(q, k2, v2)[:, :, :300])
            big = f(q * 300.0, k, v)                      # extreme scores (|s| ~ 1e3-1e4)
            edge.append(dict(dtype=dn, impl=impl, future_perturbation_invariant=same,
                             extreme_scores_finite=bool(torch.isfinite(big).all())))
        except RuntimeError as e:
            edge.append(dict(dtype=dn, impl=impl, error=str(e)[:80]))
    # all-masked query row with an explicit mask (dense vs SDPA)
    mask = torch.tril(torch.ones(L, L, dtype=torch.bool, device=DEV)); mask[7] = False
    o_d = A.dense_attention(q, k, v, causal=False, mask=mask)
    for be in ("math", "efficient", "default"):
        try:
            o_s = A.sdpa_attention(q, k, v, be, causal=False, mask=mask)
            edge.append(dict(dtype=dn, impl=f"sdpa_{be}", all_masked_row_output="NaN" if torch.isnan(o_s[:, :, 7]).any() else f"max|o|={float(o_s[:, :, 7].abs().max()):.3g}"))
        except RuntimeError as e:
            edge.append(dict(dtype=dn, impl=f"sdpa_{be}", all_masked_row_output=f"unsupported {str(e)[:60]}"))
    edge.append(dict(dtype=dn, impl="dense", all_masked_row_output=f"max|o|={float(o_d[:, :, 7].abs().max()):.3g} (defined as 0)"))
edge_df = pd.DataFrame(edge)
edge_df.to_csv(OUT / "causality_edge_cases.csv", index=False)
edge_df

## A2 tile-size sweep (exactness, intermediate storage, time)

In [ ]:
tile_rows = []
L = 2048
q32, k32, v32 = make_qkv(L, torch.float32)
ref = A.dense_attention(q32.double(), k32.double(), v32.double())
for dn, dt in DTYPES.items():
    q, k, v = [t.to(dt) for t in (q32, k32, v32)]
    for bk in (32, 64, 128, 256, 512):
        reset_peak()
        base = torch.cuda.memory_allocated()
        with torch.no_grad():
            o = A.tiled_attention(q, k, v, block_k=bk)
        peak = torch.cuda.max_memory_allocated() - base
        t = benchmark(lambda: A.tiled_attention(q, k, v, block_k=bk), warmup=2, reps=10)
        tile_rows.append(dict(dtype=dn, block_k=bk, out_max_abs=float((o.double() - ref).abs().max()),
                              fwd_ms=t["mean"], peak_above_inputs_mb=peak / 2**20,
                              theoretical_tile_scores_mb=A.theoretical_intermediate_bytes("tiled", B, H, L, D, bk) / 2**20))
del q32, k32, v32, ref; free()
tile_df = pd.DataFrame(tile_rows)
tile_df.to_csv(OUT / "tiled_block_sweep.csv", index=False)
tile_df

## Benchmark: forward-only and forward+backward latency and memory

In [ ]:
bench_path = OUT / "benchmark.jsonl"
done = {(r["impl"], r["dtype"], r["L"], r["mode"]) for r in read_jsonl(bench_path)} if bench_path.exists() else set()
from shared.src.io import append_jsonl
for L in LENGTHS:
    for dn, dt in DTYPES.items():
        for impl in IMPLS:
            for mode in ("forward", "forward+backward"):
                if (impl, dn, L, mode) in done:
                    continue
                row = dict(impl=impl, dtype=dn, L=L, mode=mode, B=B, H=H, d=D, reps=10, warmup=3, unit="ms")
                try:
                    q, k, v = [t.requires_grad_(mode != "forward") for t in make_qkv(L, dt)]
                    f = A.get_impl(impl)
                    go = torch.randn_like(q)
                    if mode == "forward":
                        def run():
                            with torch.no_grad():
                                return f(q, k, v)
                    else:
                        def run():
                            q.grad = k.grad = v.grad = None
                            f(q, k, v).backward(go)
                    free(); reset_peak()
                    base = torch.cuda.memory_allocated()
                    r = benchmark(run, warmup=3, reps=10)
                    pm = peak_memory()
                    row.update(ms=r["mean"], std=r["std"], median=r["median"], peak_alloc_mb=pm["peak_allocated_mb"],
                               peak_reserved_mb=pm["peak_reserved_mb"], peak_above_inputs_mb=(torch.cuda.max_memory_allocated() - base) / 2**20,
                               status="ok")
                    if impl == "sdpa_default":
                        row["dispatched"] = A.detect_sdpa_backend(q, k, v, "default", grad=mode != "forward")
                except torch.cuda.OutOfMemoryError:
                    row["status"] = "OOM"
                except RuntimeError as e:
                    row["status"] = f"unsupported: {str(e)[:80]}"
                append_jsonl(bench_path, row)
                q = k = v = None; free()
        print("done L =", L, flush=True)
bench = pd.DataFrame(read_jsonl(bench_path))
bench.pivot_table(index=["impl", "dtype"], columns=["mode", "L"], values="ms").round(2)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
for c, mode in enumerate(("forward", "forward+backward")):
    for dn, ls in (("fp16", "-"), ("fp32", "--")):
        sub = bench[(bench["mode"] == mode) & (bench["dtype"] == dn) & (bench["status"] == "ok")]
        for impl, g in sub.groupby("impl"):
            axes[0, c].plot(g["L"], g["ms"], ls, marker="o", label=f"{impl} {dn}")
            axes[1, c].plot(g["L"], g["peak_above_inputs_mb"], ls, marker="o", label=f"{impl} {dn}")
    axes[0, c].set_title(f"{mode} latency (B=2, H=4, d=64; mean of 10)"); axes[0, c].set_ylabel("ms")
    axes[1, c].set_title(f"{mode} peak allocated above inputs"); axes[1, c].set_ylabel("MiB")
    for r in (0, 1):
        axes[r, c].set_xscale("log", base=2); axes[r, c].set_yscale("log"); axes[r, c].set_xlabel("sequence length L")
axes[0, 1].legend(fontsize=7, ncol=2)
save_figure(ctx.figures, "attention_latency_memory_vs_length")

In [ ]:
status = bench.pivot_table(index=["impl", "dtype"], columns=["mode", "L"], values="status", aggfunc="first")
status.to_csv(OUT / "status_table.csv")
status